In [15]:
%pip install -qU langchain-groq

In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    groq_api_key="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",
    # other params...
)

In [17]:
%pip install -qU langchain_community beautifulsoup4

In [36]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/director-software-engineering-itc/job/R-49416")

In [37]:
content = loader.load().pop().page_content

In [39]:
# print(content)

In [40]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("Tell me a joke about {topic}")

prompt_template.invoke({"topic": "cats"})

StringPromptValue(text='Tell me a joke about cats')

In [44]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
    )
chain_extract = prompt_template | llm
res = chain_extract.invoke({"page_data": content})
print("Result : ",res)
print(res.content)
type(res.content)
# prompt_template.invoke({"topic": "cats"})

Result :  content='```json\n{\n    "role": "Director Software Engineering, ITC",\n    "experience": "14+ years experience leading high performing software engineering teams",\n    "skills": [\n        "Cloud native technologies",\n        "Cloud providers like AWS, Azure",\n        "Agile software development methodology",\n        "DevOps (CICD)",\n        "Modern software architectural principles",\n        "Front-end web application tech (e.g. javascript, CSS, html5, Vue, React/redux)",\n        "Distributed cloud systems tech (e.g. node.js, EC2, Kubernetes, Lambda, Bedrock, DynamoDB, Elasticsearch)",\n        "AIicense, Machine Learning and related data solutions",\n        "Microservices and applications"\n    ],\n    "description": "Lead teams that build and develop web experiences, core platform APIs leveraging AI/ML for Nike’s Merchandising functions. Partner and collaborate with business teams, other technology leaders, engineers, architects, and product managers to drive rapi

str

In [45]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Director Software Engineering, ITC',
 'experience': '14+ years experience leading high performing software engineering teams',
 'skills': ['Cloud native technologies',
  'Cloud providers like AWS, Azure',
  'Agile software development methodology',
  'DevOps (CICD)',
  'Modern software architectural principles',
  'Front-end web application tech (e.g. javascript, CSS, html5, Vue, React/redux)',
  'Distributed cloud systems tech (e.g. node.js, EC2, Kubernetes, Lambda, Bedrock, DynamoDB, Elasticsearch)',
  'AIicense, Machine Learning and related data solutions',
  'Microservices and applications'],
 'description': 'Lead teams that build and develop web experiences, core platform APIs leveraging AI/ML for Nike’s Merchandising functions. Partner and collaborate with business teams, other technology leaders, engineers, architects, and product managers to drive rapid value delivery of reusable, well-architected and highly scalable solutions to modernize Nike’s technology landscape.

In [46]:
type(json_res)

dict

In [47]:
job = json_res

In [48]:
job

{'role': 'Director Software Engineering, ITC',
 'experience': '14+ years experience leading high performing software engineering teams',
 'skills': ['Cloud native technologies',
  'Cloud providers like AWS, Azure',
  'Agile software development methodology',
  'DevOps (CICD)',
  'Modern software architectural principles',
  'Front-end web application tech (e.g. javascript, CSS, html5, Vue, React/redux)',
  'Distributed cloud systems tech (e.g. node.js, EC2, Kubernetes, Lambda, Bedrock, DynamoDB, Elasticsearch)',
  'AIicense, Machine Learning and related data solutions',
  'Microservices and applications'],
 'description': 'Lead teams that build and develop web experiences, core platform APIs leveraging AI/ML for Nike’s Merchandising functions. Partner and collaborate with business teams, other technology leaders, engineers, architects, and product managers to drive rapid value delivery of reusable, well-architected and highly scalable solutions to modernize Nike’s technology landscape.

In [28]:
import pandas as pd
df = pd.read_csv("/content/my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [30]:
# ! pip install -qU chromadb

In [32]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')

# "PersistentClient" : create a db and writes it on the disk
# "Client" : creates a db in the memory and the memory will be erased on restart

collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 105MiB/s]


In [33]:
links = collection.query(query_texts=["Experienced in python,expertise in react"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}]]

In [49]:
job

{'role': 'Director Software Engineering, ITC',
 'experience': '14+ years experience leading high performing software engineering teams',
 'skills': ['Cloud native technologies',
  'Cloud providers like AWS, Azure',
  'Agile software development methodology',
  'DevOps (CICD)',
  'Modern software architectural principles',
  'Front-end web application tech (e.g. javascript, CSS, html5, Vue, React/redux)',
  'Distributed cloud systems tech (e.g. node.js, EC2, Kubernetes, Lambda, Bedrock, DynamoDB, Elasticsearch)',
  'AIicense, Machine Learning and related data solutions',
  'Microservices and applications'],
 'description': 'Lead teams that build and develop web experiences, core platform APIs leveraging AI/ML for Nike’s Merchandising functions. Partner and collaborate with business teams, other technology leaders, engineers, architects, and product managers to drive rapid value delivery of reusable, well-architected and highly scalable solutions to modernize Nike’s technology landscape.

In [50]:
job["skills"]

['Cloud native technologies',
 'Cloud providers like AWS, Azure',
 'Agile software development methodology',
 'DevOps (CICD)',
 'Modern software architectural principles',
 'Front-end web application tech (e.g. javascript, CSS, html5, Vue, React/redux)',
 'Distributed cloud systems tech (e.g. node.js, EC2, Kubernetes, Lambda, Bedrock, DynamoDB, Elasticsearch)',
 'AIicense, Machine Learning and related data solutions',
 'Microservices and applications']

In [51]:
links = collection.query(query_texts=job["skills"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/typescript-frontend-portfolio'},
  {'links': 'https://example.com/full-stack-js-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/react-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}]]

Email generation:

In [56]:
prompt_email = PromptTemplate.from_template(
    """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are varadaraj kamisetty, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are varadaraj kamisetty, BDE at AtliQ.
        Do provide a neat and clean Email and do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Software Engineering Solutions for Nike's Merchandising Functions

Dear Hiring Manager,

I came across the job description for Director Software Engineering, ITC at Nike, and I am excited to introduce AtliQ, an AI & Software Consulting company that can help you achieve your goals. With 14+ years of experience leading high-performing software engineering teams, I believe our expertise aligns perfectly with your requirements.

At AtliQ, we have a proven track record of delivering scalable, well-architected, and highly scalable solutions that modernize technology landscapes. Our expertise in cloud native technologies, cloud providers like AWS and Azure, agile software development methodology, DevOps (CICD), and modern software architectural principles can help drive rapid value delivery for Nike's Merchandising functions.

Our team is well-versed in front-end web application technologies such as JavaScript, CSS, HTML5, Vue, and React/Redux, as well as distributed cloud sys